In [ ]:
from proto import *
from engine import *
from utils import *

import pandas as pd
import numpy as np
import datetime as dt

pd.options.plotting.backend = "plotly"

In [ ]:
import computegraph as cg

In [ ]:
class Parameter(cg.Variable):
    def __init__(self, key, default):
        super().__init__(key, "parameters")
        self.default = default


class ModelVariable(cg.Variable):
    def __init__(self, key):
        super().__init__(key, "model_variables")
        self.key = key


Time = ModelVariable("time")
param = Parameter


def defer(func, name=None):
    def _proxy(*args, **kwargs):
        return cg.Function(func, args, kwargs, name)

    return _proxy


def label(graph_obj, name):
    graph_obj.node_name = name
    return graph_obj

In [ ]:
from utils import LinearInterpolator

In [ ]:
from computegraph.types import Data

In [ ]:
epoch = Epoch(dt.datetime(1980, 6, 7))

In [ ]:
x = jnp.array([0.0, 15.0, 20.0, 40.0])
y = jnp.array([0.0, 0.0, 2.0, 0.0])

t = LinearInterpolator(x, y)

t.process(jnp.linspace(0.0, 50.0))

In [ ]:
double_time = defer(lambda t: t * 2.0, "doubletime")(Time * 2.0)
vinterp = defer(LinearInterpolator, "CRInterpolator")(
    Data(x, "cr_times"), Data(y, "cr_values")
)

In [ ]:
vacct = defer(LinearInterpolator.process)(vinterp, Time)

In [ ]:
disease_state = Stratification("disease_state", ["S", "I", "R"])
humans = CompartmentMap.new(disease_state)
humans.compartments

In [ ]:
age_strat = humans.stratify(Stratification("age", ["child", "adult"]))

In [ ]:
humans.compartments

In [ ]:
infection = TransitionFlow(
    disease_state["S"], disease_state["I"], vacct * Parameter("contact_rate", 0.2)
)
recovery = TransitionFlow(
    disease_state["I"], disease_state["R"], Parameter("recovery_rate", 0.1)
)

In [ ]:
infection

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {}  # {"foi": foi}

model = GraphModel(humans, flows, dyn_params)
runner = model.get_runner()

istate = humans.zeros(np).data
S_idx = humans.query(disease_state["S"]).indices
istate[S_idx] = [50.0, 100.0]

params = {"contact_rate": 0.2, "recovery_rate": 0.1}
t = 50

comp_results = runner.run(istate, params, t)

In [ ]:
runner.model.actual_flows["infection"].src_cmap

In [ ]:
runner.graph.draw()

In [ ]:
def compname(c: Compartment):
    return "_".join([stratum for (strat, stratum) in c.strata])


comp_labels = [compname(c) for c in model.cmap.compartments]

pd.DataFrame(comp_results["compartments"], columns=comp_labels).plot()

In [ ]:
pd.DataFrame(comp_results["flows"]["infection"]).plot()

In [ ]:
iflow_data = comp_results["flows"]["infection"]

In [ ]:
iflow_data.shape

In [ ]:
from experimental import ManagedArray

In [ ]:
ma = ManagedArray(
    ctvals,
    dims=["time", "compartment"],
    indices={"time": ("time", dti), "compartment": ("compartment", cmap)},
)

In [ ]:
from utils import Epoch

In [ ]:
adti

In [ ]:
ManagedArray(iflow_data, ["time", "flow"], indices={"time": dti})

In [ ]:
class CompartmentalEpiModel:
    def __init__(self, base_compartments):
        base_strat = Stratification("base", base_compartments)
        self.compartment_map = CompartmentMap.new(base_strat)
        self.flows = {}

    def add_transition_flow(self, name, source, dest, param):
        flow = TransitionFlow(source, dest, param)
        self.flows[name] = flow

    def add_infection_frequency_flow(self, name, source, dest):
        self.add_transition_flow(name, source, dest, "foi")

    def stratify(self, stratification, stratifies, mm=None):
        self.compartment_map.stratify(stratification, stratifies)
        if mm is not None:
            self.mm = mm

mm = jnp.ones((len(age_strat.strata), len(age_strat.strata)))


    def foi_mixing(self, cdatamap, params):
        ipops = query_cat_reduction(inf_age_cats, cdatamap)
        total_pop = cdatamap.data.sum()
        age_foi = mm @ ipops / total_pop * params["contact_rate"]
        return CategoryData(age_strat.categories(), age_foi)
        

In [ ]:
infection = TransitionFlow(disease_state["S"], disease_state["I"], "foi")
recovery = TransitionFlow(disease_state["I"], disease_state["R"], "recovery_rate")

In [ ]:
def foi(cdatamap, params):
    ipop = cdatamap.query([disease_state["I"]]).data.sum()
    total_pop = cdatamap.data.sum()
    return (ipop / total_pop) * params["contact_rate"]

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {"foi": foi}

model = NaiveModel(humans, flows, dyn_params)
run = model.get_runner()

istate = jnp.array([100.0, 5.0, 0.0])
params = {"contact_rate": 0.2, "recovery_rate": 0.01}
t = 200

comp_results = run(istate, params, t)

In [ ]:
severity_strat = humans.stratify(
    Stratification("severity", ["mild", "severe"]), disease_state["I"]
)

In [ ]:
humans

In [ ]:
run = model.get_runner()
istate = jnp.array([100.0, 5.0, 0.0, 0.0])
params = {"contact_rate": 0.2, "recovery_rate": 0.01}
t = 200

comp_results = run(istate, params, t)

comp_labels = [compname(c) for c in model.cmap.compartments]
pd.DataFrame(comp_results, columns=comp_labels).plot()

In [ ]:
age_strat = humans.stratify(
    Stratification("age", ["child", "young_adult", "adult", "older"])
)

In [ ]:
inf_age_cats = [
    [disease_state["I"], age_strat[age_group]] for age_group in age_strat.strata
]

inf_age_cats

In [ ]:
mm = jnp.ones((len(age_strat.strata), len(age_strat.strata)))


def foi_mixing(cdatamap, params):
    ipops = query_cat_reduction(inf_age_cats, cdatamap)
    total_pop = cdatamap.data.sum()
    age_foi = mm @ ipops / total_pop * params["contact_rate"]
    return CategoryData(age_strat.categories(), age_foi)

In [ ]:
istate = humans.zeros(np)
s_idx = humans.query(disease_state["S"]).indices
istate.data[s_idx] = np.array((10.0, 20.0, 50.0, 20.0))
i_idx = humans.query(disease_state["I"]).indices
istate.data[i_idx] = 1.0

In [ ]:
foi_mixing(istate, params)

In [ ]:
flows = {"infection": infection, "recovery": recovery}

dyn_params = {"foi": foi_mixing}

model = NaiveModel(humans, flows, dyn_params)

run = model.get_runner()

params = {"contact_rate": 0.2, "recovery_rate": 0.02}
t = 200

comp_results = run(istate.data, params, t)

comp_labels = [compname(c) for c in model.cmap.compartments]
pd.DataFrame(comp_results, columns=comp_labels).plot()